# Predicting CANDOR survey responses based on growth rates $\alpha$

In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

from scipy.stats import chisquare, skewtest, t

# stats in R
import rpy2.robjects as ro
from rpy2.robjects import Formula
from rpy2.robjects.conversion import localconverter
from rpy2.robjects import pandas2ri, default_converter, conversion


import warnings; warnings.filterwarnings('ignore')

In [ ]:
vars = [
    ### Affective motive
    'best_affect', 'conv_leader', 'conversationalist', 'how_enjoyable', 'i_like_you', 'i_felt_close_to_my_partner', 'i_think_your_status', 'i_would_like_to_become_friends', 'overall_affect',
    'responsive_1','responsive_2','responsive_3',
    'you_are_good_listener', 'you_are_kind', 'you_are_quickwitted','you_are_warm','you_like_me',
    #test affectives!
    'you_are_intelligent', 'you_are_friendly', 'in_common',

    ### Cognitive motives
    'anticipated_each_other_sr6', 'developed_joint_perspective_sr2', 'thoughts_became_more_alike_sr5',
    #test cognitives!
    'shared_reality', 'our_thoughts_synced_up_sr1', 'saw_world_in_same_way_sr8',
]

Data paths used in the project

In [ ]:
subcorpora_col = 'corpus'

SLOPES_PATH = 'reports/all.parquet'

candor_meta_data_path = '../../updated-data/all-candor-meta_data.csv'

TABLES_FIGS_PATH = 'tables_and_figures/'
if not os.path.exists(TABLES_FIGS_PATH):
    os.mkdir(TABLES_FIGS_PATH)

PARAMETER_ESTIMATES_PATH = 'params/'
if not os.path.exists(PARAMETER_ESTIMATES_PATH):
    os.mkdir(PARAMETER_ESTIMATES_PATH)

if not os.path.exists(os.path.join(TABLES_FIGS_PATH, 'individual_var_stats')):
    os.mkdir(os.path.join(TABLES_FIGS_PATH, 'individual_var_stats'))

## Importing data

In [ ]:
results = pd.read_parquet(SLOPES_PATH)
results.head()

##### Some preliminary statistics

In [ ]:
print((results['lambda'].abs() == np.inf).sum(), (results['lambda'].abs() == np.inf).mean())
# results.isna().sum()

### Removing null-test results/bad values

In [ ]:
sel = results['lambda'].isna() | (results['lambda'].abs() == np.inf)
sel.sum(), sel.mean(), (~sel).sum()

In [ ]:
# is_null = np.array(['null-' in x for x in tqdm(results['x_turn_id'])])
# is_null = results['null']
# is_null.sum()

In [ ]:
results = results.loc[
    (~sel)
    # & (~is_null)
    & results[subcorpora_col].isin(['CANDOR'])
]
results.shape

In [ ]:
results['conversation_id'] = ['-'.join(it.split('-')[3:]) for it in tqdm(results['cid'])]

In [ ]:
results.head()

## Merging with CANDOR metadata

In [ ]:
dfm = pd.read_csv(candor_meta_data_path)[['user_id', 'convo_id']+vars]
print(dfm.shape)


rescaling data to be in same scale.

In [ ]:
for col in tqdm(list(dfm)[2:]):
    dfm[col] = (dfm[col] / dfm[col].max() ) * 10

In [ ]:
dfm.head()

#### Testing for normalcy and uniformity

In [ ]:
normalcy_and_other_stats = []

In [ ]:
for var in vars:
    sel = dfm[var].isna()
    maxima = dfm[var].loc[~sel].max()
    minima = dfm[var].loc[~sel].min()
    medi = dfm[var].loc[~sel].median()
    mu = dfm[var].loc[~sel].mean()
    mode = dfm[var].loc[~sel].mode()
    skew_stat, p_skew = skewtest(dfm[var].loc[~sel].values)

    dv = dfm[var].loc[~sel].value_counts()
    chi_stat, p_chi = chisquare(f_obs=dv.values, f_exp=np.array([1/len(dv)]*len(dv))*dv.sum())

    normalcy_and_other_stats += [
        {
            'variable': var,
            'range': '[{}, {}]'.format(minima, maxima),
            'median': medi.item(),
            'mean': mu.item(),
            'mode': mode.values,
            'skew_stat': skew_stat, 'skew_p': p_skew,
            'chi_stat': chi_stat, 'chi_p': p_chi
        }
    ]

normalcy_and_other_stats = pd.DataFrame(normalcy_and_other_stats)

In [ ]:
normalcy_and_other_stats.head(n=40)

In [ ]:
normalcy_and_other_stats.to_csv('candor-normalcy-stats.csv', index=False, encoding='utf-8')

#### Merging docs

In [ ]:
results = pd.merge(
    left=results, left_on=['x_speaker', 'conversation_id'],
    right=dfm, right_on=['user_id', 'convo_id'],
    # left=results, left_on=['speaker'],
    # right=dfm, right_on=['user_id'],
    how='left'
)

In [ ]:
results = results.drop(columns=['user_id', 'convo_id'])
# results = results.drop(columns=['user_id'])
# results.columns
results.isna().sum()

## Predicting individual survey variables from decay rates $\alpha$

### Converting categorical cols to numbered indexes

In [ ]:
## convert speaker to integers
col = 'x_speaker'
vs = results[col].unique()
# sid = {sp:i+1 for i,sp in enumerate(np.random.choice(vs,size=(len(vs),),replace=False))}
sid = {sp:i+1 for i,sp in enumerate(vs)}
results[col+'_'] = [sid[sp] for sp in tqdm(results[col])]

In [ ]:
## convert turn_ids to integers
col = 'conversation_id'
vs = results[col].astype(str).unique()
# sid = {sp:i+1 for i,sp in enumerate(np.random.choice(vs,size=(len(vs),),replace=False))}
sid = {sp:i+1 for i,sp in enumerate(vs)}
results[col+'_'] = [sid[sp] for sp in tqdm(results[col].astype(str))]

### Running JAGS model in R

In [ ]:
# model_ = str(model_script)

for var in tqdm(vars):
    sel_2 = ~results[var].isna()

    with localconverter(default_converter + pandas2ri.converter):
        df_ = conversion.py2rpy(results[['lambda', 'conversation_id_', 'x_speaker_', var]].loc[sel_2])

    ro.globalenv['df'] = df_

    ro.r("""
    library(lme4)
    # print(colnames(df))
    model <- lmer()
    survey_var <- df${}
    slope <- df$lambda
    speaker <- df$x_speaker_
    conversation <- df$conversation_id_
    KSPEAKER <- max(speaker)
    CCONVERSATION <- max(conversation)
    # print(KSPEAKER)
    # print(CCONVERSATION)
    ROWS <- length(slope)

    b0_mu <- mean(slope)
    lv <- var(slope)

    data <- list(
        'lv',
        'survey_var',
        'slope',
        # 'speaker',
        # 'conversation',
        'b0_mu',
        # 'CCONVERSATION',
        # 'KSPEAKER',
        'ROWS'
    )

    myinits <- list(
        list(
            eta=.5,
            # b0=0.1,
            # phi=0.1,
            beta=0.01
        )
    )

    parameters <- c(
        'beta',
        'eta',
        'b0'
    )

    samples<- jags(data, inits=myinits, parameters, model.file='model.txt', n.chains=1, n.iter=5000, n.burnin=100, DIC=T, progress.bar="text")

    beta <- samples$BUGSoutput$sims.list$beta
    b0 <- samples$BUGSoutput$sims.list$b0
    eta <- samples$BUGSoutput$sims.list$eta
    save(
        beta,
        b0,
        eta,
        file='{}/{}.RData'
    )
    """.format(var, PARAMETER_ESTIMATES_PATH, var))